## Caso não tenha as libs instaladas no Kernel

In [1]:
# %pip install plotly pandas scikit-learn opencv-python
# %pip install --upgrade nbformat

## Import das libs

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import json
from pathlib import Path

from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Analise dos logs de voo em memmap

In [3]:
lista_dfs = []
caminhos_ficheiros = []


def listar_logs_voo_memmap():
    candidatos = sorted(Path("../logs").glob("voo_teste_*/manifest.json"))
    if not candidatos:
        candidatos = sorted(Path("logs").glob("voo_teste_*/manifest.json"))
    return candidatos

def carregar_log_voo_memmap(manifest_path):
    manifest_path = Path(manifest_path)

    try:
        texto = manifest_path.read_text(encoding="utf-8").strip()

        if not texto:
            print(f"Ignorando manifest vazio: {manifest_path}")
            return pd.DataFrame()

        manifest = json.loads(texto)

    except JSONDecodeError as e:
        print(f"Ignorando manifest inválido: {manifest_path}")
        print(f"Erro: {e}")
        return pd.DataFrame()

    n = int(manifest.get("num_samples", 0))
    if n <= 0:
        return pd.DataFrame()

    intervalos_path = manifest_path.parent / manifest["arrays"]["flight_intervals"]

    if not intervalos_path.exists():
        print(f"Ignorando run sem memmap: {intervalos_path}")
        return pd.DataFrame()

    intervalos = np.load(intervalos_path, mmap_mode="r")[:n]

    df = pd.DataFrame(intervalos)
    df["run_id"] = manifest_path.parent.name
    df["manifest_path"] = str(manifest_path)

    df["tempo_s"] = df["dt_s"].fillna(0).cumsum()
    df["x_rel"] = df["delta_x_m"].fillna(0).cumsum()
    df["y_rel"] = df["delta_y_m"].fillna(0).cumsum()
    df["z_rel"] = df["delta_z_m"].fillna(0).cumsum()
    df["altitude_rel"] = -df["z_rel"]

    return df

for manifest_path in listar_logs_voo_memmap():
    df_temp = carregar_log_voo_memmap(manifest_path)
    if not df_temp.empty:
        caminhos_ficheiros.append(manifest_path)
        lista_dfs.append(df_temp)

if not lista_dfs:
    display(Markdown("Nenhum log novo em memmap encontrado em `logs/voo_teste_*/manifest.json`."))

### Deslocamento acumulado relativo

In [ ]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_3d = go.Figure()
    fig_3d.add_trace(go.Scatter3d(x=df["x_rel"], y=df["y_rel"], z=df["altitude_rel"], mode="lines", line=dict(color="royalblue", width=4), name="Deslocamento acumulado"))
    fig_3d.add_trace(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="green", size=6), name="Origem relativa"))
    fig_3d.add_trace(go.Scatter3d(x=[df["x_rel"].iloc[-1]], y=[df["y_rel"].iloc[-1]], z=[df["altitude_rel"].iloc[-1]], mode="markers", marker=dict(color="red", size=6, symbol="x"), name="Fim relativo"))
    fig_3d.update_layout(title=f"Deslocamento acumulado por deltas - Run: {timestamp}", scene=dict(xaxis_title="Delta X acumulado (m)", yaxis_title="Delta Y acumulado (m)", zaxis_title="Delta altitude acumulada (m)", camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))), legend=dict(x=0, y=1), margin=dict(l=0, r=0, b=0, t=40))
    fig_3d.show()

### Variacao das velocidades angulares

In [ ]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_2d = go.Figure()
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_roll_speed_rad_s"], mode="lines", name="Delta roll speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_pitch_speed_rad_s"], mode="lines", name="Delta pitch speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_yaw_speed_rad_s"], mode="lines", name="Delta yaw speed", opacity=0.7))
    fig_2d.update_layout(title=f"Variacao das velocidades angulares - Run: {timestamp}", xaxis_title="Tempo acumulado por intervalos (s)", yaxis_title="Delta velocidade angular (rad/s)", template="plotly_white", hovermode="x unified")
    fig_2d.show()

### Analise dos intervalos depth/flow em memmap

Esta analise usa as novas runs em `datasets/depth_ground_truth/run_*/manifest.json`. Cada amostra representa a variacao entre duas atualizacoes consecutivas da logica de proximidade visual por optical flow, a mesma logica que gera as flechas desenhadas na deteccao de obstaculos.

O foco deixa de ser o estado absoluto do drone em um frame isolado e passa a ser o deslocamento sincronizado do intervalo: deltas de posicao, atitude, IMU, comandos reativos, flow e depth ground truth. O objetivo e deixar os dados menos dependentes da origem da simulacao e mais genericos para treino e analise.

In [ ]:
# Funcoes para leitura das novas runs depth/flow em numpy.memmap
def localizar_raiz_projeto_memmap(nome_dataset="datasets"):
    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / nome_dataset / "depth_ground_truth").exists():
            return candidato
    return Path.cwd()


def listar_runs_depth_memmap(base_dir):
    depth_dir = base_dir / "datasets" / "depth_ground_truth"
    return sorted(path.parent for path in depth_dir.glob("run_*/manifest.json"))


def carregar_manifesto_memmap(run_dir):
    with open(run_dir / "manifest.json", encoding="utf-8") as fp:
        return json.load(fp)


def abrir_array_memmap(run_dir, manifesto, chave):
    return np.load(run_dir / manifesto["arrays"][chave], mmap_mode="r")


def indices_intervalos_sincronizados(intervalos, max_age_s=0.08, max_interval_s=0.5):
    if intervalos.empty:
        return np.array([], dtype=int)
    dt_rgb = pd.to_numeric(intervalos["dt_s"], errors="coerce")
    dt_depth = pd.to_numeric(intervalos["depth_dt_s"], errors="coerce")
    depth_age = pd.to_numeric(intervalos["depth_age_s"], errors="coerce")
    mascara = (
        dt_rgb.notna() & dt_rgb.gt(0.0) & dt_rgb.le(max_interval_s)
        & dt_depth.notna() & dt_depth.gt(0.0) & dt_depth.le(max_interval_s)
        & depth_age.notna() & depth_age.le(max_age_s)
    )
    return np.flatnonzero(mascara.to_numpy())


def carregar_run_depth_memmap(run_dir):
    manifesto = carregar_manifesto_memmap(run_dir)
    n = int(manifesto.get("num_samples", 0))
    intervalos_mm = abrir_array_memmap(run_dir, manifesto, "intervals")
    intervalos_brutos = pd.DataFrame.from_records(intervalos_mm[:n]).copy()
    indices_validos = indices_intervalos_sincronizados(intervalos_brutos)
    intervalos = intervalos_brutos.iloc[indices_validos].reset_index(drop=True)
    if not intervalos.empty:
        intervalos["run_id"] = run_dir.name
        intervalos["ordem_intervalo"] = np.arange(len(intervalos))
    return {
        "run_dir": run_dir,
        "manifesto": manifesto,
        "intervalos": intervalos,
        "image_delta_bgr": abrir_array_memmap(run_dir, manifesto, "image_delta_bgr")[indices_validos],
        "depth_delta_log": abrir_array_memmap(run_dir, manifesto, "depth_delta_log")[indices_validos],
        "depth_delta_mask": abrir_array_memmap(run_dir, manifesto, "depth_delta_mask")[indices_validos],
        "flow_vectors": abrir_array_memmap(run_dir, manifesto, "flow_vectors")[indices_validos],
    }


def carregar_todas_runs_depth_memmap(base_dir):
    runs = []
    for run_dir in listar_runs_depth_memmap(base_dir):
        try:
            run = carregar_run_depth_memmap(run_dir)
        except Exception as exc:
            print(f"Run ignorada em {run_dir.name}: {exc}")
            continue
        if len(run["intervalos"]) > 0:
            runs.append(run)
    return runs

In [ ]:
raiz_projeto = localizar_raiz_projeto_memmap()
runs_depth_memmap = carregar_todas_runs_depth_memmap(raiz_projeto)

if not runs_depth_memmap:
    display(Markdown(
        "Nenhuma run nova em memmap encontrada em `datasets/depth_ground_truth/run_*/manifest.json`. "
        "Colete uma nova run com `save_ground_truth_dataset:=true`."
    ))
else:
    depth_run = runs_depth_memmap[-1]
    depth_df = depth_run["intervalos"].copy()
    manifesto = depth_run["manifesto"]
    print(f"Run analisada: {depth_run['run_dir'].name}")
    print(f"Intervalos depth/flow validos: {len(depth_df)}")
    print(f"Schema: {manifesto.get('schema_version')}")
    colunas_resumo = [
        "dt_s", "depth_age_s", "depth_dt_s", "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad", "flow_valid_points",
        "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px",
        "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp",
    ]
    display(depth_df[[c for c in colunas_resumo if c in depth_df.columns]].describe().T)
    fig_depth_delta = go.Figure()
    for coluna, nome in [("delta_depth_p10_m", "Delta depth P10"), ("delta_depth_p50_m", "Delta depth P50"), ("delta_depth_close_5m_pp", "Delta pixels < 5 m")]:
        if coluna in depth_df.columns:
            fig_depth_delta.add_trace(go.Scatter(x=depth_df["ordem_intervalo"], y=depth_df[coluna], mode="lines+markers", name=nome))
    fig_depth_delta.update_layout(title="Variacao de profundidade/proximidade por intervalo visual", xaxis_title="Intervalo visual em ordem de coleta", yaxis_title="Delta do intervalo", template="plotly_white", hovermode="x unified")
    fig_depth_delta.show()
    fig_flow_depth = px.scatter(depth_df, x="flow_mag_p90_px", y="delta_depth_close_5m_pp", color="radial_flow_p90_px", size="flow_valid_points", hover_data=["sample_id", "dt_s", "delta_x_m", "delta_yaw_heading_rad"], title="Flow visual x variacao de ocupacao proxima", template="plotly_white")
    fig_flow_depth.show()
    ranking = depth_df.assign(impacto_proximidade=depth_df["delta_depth_close_5m_pp"].abs()).sort_values(["impacto_proximidade", "flow_mag_p90_px"], ascending=False)
    display(Markdown("**Intervalos mais informativos para inspecao/treino:**"))
    display(ranking[["run_id", "sample_id", "dt_s", "flow_valid_points", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad"]].head(12))

### Baseline MLP com vetores de variacao

A MLP agora trabalha com features tabulares de deslocamento entre intervalos visuais: deltas de estado, IMU, comandos, estatisticas do optical flow e estatisticas da diferenca de imagem estabilizada. Os alvos tambem sao deltas de depth/proximidade, e nao profundidades absolutas.

A comparacao contra `DummyRegressor(strategy="mean")` continua sendo usada como referencia minima: a MLP so e util se aprender uma relacao melhor que prever a variacao media observada no treino.

In [ ]:
TARGETS_MLP_DELTA = ["delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_p90_m", "delta_depth_close_2m_pp", "delta_depth_close_5m_pp", "delta_depth_close_10m_pp"]


def features_image_delta(img):
    arr = np.asarray(img, dtype=np.float32)
    abs_arr = np.abs(arr)
    return {"img_delta_mean": float(arr.mean()), "img_delta_std": float(arr.std()), "img_delta_abs_mean": float(abs_arr.mean()), "img_delta_abs_p90": float(np.percentile(abs_arr, 90))}


def features_flow_vectors(vetores, valid_points):
    n = int(max(0, min(valid_points, len(vetores))))
    if n == 0:
        return {"flow_vec_rel_std": 0.0, "flow_vec_xy_mean": 0.0, "flow_vec_radial_mean": 0.0, "flow_vec_risk_mean": 0.0}
    v = np.asarray(vetores[:n], dtype=np.float32)
    return {"flow_vec_rel_std": float(v[:, :2].std()), "flow_vec_xy_mean": float(v[:, 2:4].mean()), "flow_vec_radial_mean": float(v[:, 4].mean()), "flow_vec_risk_mean": float(v[:, 5].mean())}


def montar_dataset_mlp_delta_depth(base_dir):
    linhas_x, linhas_y, linhas_meta = [], [], []
    for run in carregar_todas_runs_depth_memmap(base_dir):
        df = run["intervalos"].reset_index(drop=True)
        for i, row in df.iterrows():
            if any(col not in row.index or not np.isfinite(row[col]) for col in TARGETS_MLP_DELTA):
                continue
            feats = {}
            for coluna in df.select_dtypes(include=[np.number]).columns:
                if coluna in TARGETS_MLP_DELTA or coluna.startswith("delta_depth_") or coluna in {"delta_valid_px_pct", "sample_id", "ordem_intervalo"}:
                    continue
                valor = row[coluna]
                feats[coluna] = 0.0 if pd.isna(valor) else float(valor)
            feats.update(features_image_delta(run["image_delta_bgr"][i]))
            feats.update(features_flow_vectors(run["flow_vectors"][i], row.get("flow_valid_points", 0)))
            linhas_x.append(feats)
            linhas_y.append({alvo: float(row[alvo]) for alvo in TARGETS_MLP_DELTA})
            linhas_meta.append({"run_id": run["run_dir"].name, "sample_id": int(row.get("sample_id", i + 1)), "ordem_intervalo": int(row.get("ordem_intervalo", i))})
    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


def separar_intervalos(meta_df, random_state=42):
    n = len(meta_df)
    indices = np.arange(n)
    runs = meta_df["run_id"].dropna().unique() if n else []
    if len(runs) >= 3:
        rng = np.random.default_rng(random_state)
        runs = np.array(runs); rng.shuffle(runs)
        n_train = max(1, int(len(runs) * 0.7)); n_val = max(1, int(len(runs) * 0.15))
        train_runs = set(runs[:n_train]); val_runs = set(runs[n_train:n_train+n_val]); test_runs = set(runs[n_train+n_val:]) or {runs[-1]}
        train_runs -= test_runs
        return {"train": indices[meta_df["run_id"].isin(train_runs).to_numpy()], "val": indices[meta_df["run_id"].isin(val_runs).to_numpy()], "test": indices[meta_df["run_id"].isin(test_runs).to_numpy()], "modo": "por run_id"}
    n_train = max(1, int(n * 0.70)); n_val = max(1, int(n * 0.15))
    return {"train": indices[:n_train], "val": indices[n_train:n_train+n_val], "test": indices[n_train+n_val:], "modo": "temporal por intervalos"}


def avaliar_delta(y_true, y_pred, targets, modelo, split):
    linhas = []
    for i, alvo in enumerate(targets):
        real = np.asarray(y_true[:, i], dtype=float); pred = np.asarray(y_pred[:, i], dtype=float)
        corr = float(np.corrcoef(real, pred)[0, 1]) if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9 else np.nan
        linhas.append({"modelo": modelo, "split": split, "alvo_delta": alvo, "MAE": float(mean_absolute_error(real, pred)), "RMSE": float(mean_squared_error(real, pred) ** 0.5), "corr": corr})
    return linhas


def treinar_avaliar_mlp_delta_depth(X, y, meta_df):
    targets = list(y.columns); split = separar_intervalos(meta_df)
    if len(split["test"]) == 0: split["test"] = split["val"]
    if len(split["val"]) == 0: split["val"] = split["test"]
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    mlp = TransformedTargetRegressor(regressor=Pipeline([("x_scaler", StandardScaler()), ("mlp", MLPRegressor(hidden_layer_sizes=(64, 16), activation="relu", solver="lbfgs", alpha=0.01, max_iter=2000, random_state=42))]), transformer=StandardScaler())
    dummy = DummyRegressor(strategy="mean")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); mlp.fit(Xv[split["train"]], yv[split["train"]])
    dummy.fit(Xv[split["train"]], yv[split["train"]])
    linhas, predicoes = [], {}
    for nome_split, idx in [("validacao", split["val"]), ("teste", split["test"])]:
        pred_mlp = mlp.predict(Xv[idx]); pred_dummy = dummy.predict(Xv[idx])
        predicoes[nome_split] = {"idx": idx, "real": yv[idx], "mlp": pred_mlp, "dummy": pred_dummy}
        linhas.extend(avaliar_delta(yv[idx], pred_mlp, targets, "MLP", nome_split)); linhas.extend(avaliar_delta(yv[idx], pred_dummy, targets, "Media treino", nome_split))
    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


raiz_mlp = localizar_raiz_projeto_memmap()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)
if len(X_mlp) < 12:
    display(Markdown("Dataset insuficiente para treinar a MLP de deltas. Colete mais intervalos depth/flow em memmap."))
else:
    modelo_mlp_delta, baseline_media_delta, split_mlp, resultados_mlp_delta, predicoes_mlp_delta = treinar_avaliar_mlp_delta_depth(X_mlp, y_mlp, meta_mlp)
    display(Markdown(f"**Dataset MLP delta:** {len(X_mlp)} intervalos, {X_mlp.shape[1]} features de variacao, {y_mlp.shape[1]} alvos delta. Separacao: {split_mlp['modo']}. Treino={len(split_mlp['train'])}, validacao={len(split_mlp['val'])}, teste={len(split_mlp['test'])}."))
    display(resultados_mlp_delta.sort_values(["split", "alvo_delta", "modelo"]).reset_index(drop=True))
    resultados_teste = resultados_mlp_delta[resultados_mlp_delta["split"] == "teste"]
    px.bar(resultados_teste, x="alvo_delta", y="MAE", color="modelo", barmode="group", title="Erro MAE no teste: MLP de deltas x media do treino", template="plotly_white").show()
    alvo_plot = "delta_depth_close_5m_pp" if "delta_depth_close_5m_pp" in y_mlp.columns else y_mlp.columns[0]
    alvo_idx = list(y_mlp.columns).index(alvo_plot); pred_teste = predicoes_mlp_delta["teste"]; meta_teste = meta_mlp.iloc[pred_teste["idx"]].reset_index(drop=True)
    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["real"][:, alvo_idx], mode="lines+markers", name="real"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["mlp"][:, alvo_idx], mode="lines+markers", name="MLP"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["dummy"][:, alvo_idx], mode="lines", name="media treino"))
    fig_pred.update_layout(title=f"Predicao no teste para {alvo_plot}", xaxis_title="Intervalos de teste em ordem temporal", yaxis_title="Delta do alvo", template="plotly_white", hovermode="x unified"); fig_pred.show()
    display(Markdown("**Intervalos de teste para inspecao:**"))
    display(pd.DataFrame({"run_id": meta_teste["run_id"], "sample_id": meta_teste["sample_id"], f"{alvo_plot}_real": pred_teste["real"][:, alvo_idx], f"{alvo_plot}_mlp": pred_teste["mlp"][:, alvo_idx], f"{alvo_plot}_baseline_media": pred_teste["dummy"][:, alvo_idx]}).head(15))

### Modelo em duas etapas: evento de proximidade + regressao nos eventos

Este experimento preserva a MLP multi-output anterior e adiciona uma avaliacao alternativa para o alvo `delta_depth_close_5m_pp`. A ideia e separar a pergunta em duas partes: primeiro detectar se houve uma mudanca relevante de proximidade e, so depois, estimar a magnitude/sinal dessa mudanca.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier

EVENT_TARGET_DELTA = "delta_depth_close_5m_pp"
EVENT_THRESHOLD_PP = 0.5
EVENT_PROBA_THRESHOLD = 0.35
EVENT_TEMPORAL_LAGS = (1, 2, 3)


def signed_log1p_array(y, scale=EVENT_THRESHOLD_PP):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y) / max(scale, 1e-6))


def signed_expm1_array(z, scale=EVENT_THRESHOLD_PP):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * np.expm1(np.abs(z)) * max(scale, 1e-6)


def colunas_temporais_evento(X_base):
    preferidas = [
        "dt_s", "depth_age_s", "depth_dt_s",
        "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad",
        "flow_valid_points", "flow_track_retention_pct",
        "flow_mag_p90_px", "radial_flow_p90_px", "pan_comp_delta_rad",
        "img_delta_abs_mean", "img_delta_abs_p90",
        "flow_vec_radial_mean", "flow_vec_risk_mean", "flow_vec_rel_std",
    ]
    return [col for col in preferidas if col in X_base.columns]


def adicionar_contexto_temporal_evento(X_base, meta_base, lags=EVENT_TEMPORAL_LAGS):
    X_base = X_base.reset_index(drop=True).copy()
    meta_ordem = meta_base.reset_index(drop=True).copy()
    if "ordem_intervalo" not in meta_ordem.columns:
        meta_ordem["ordem_intervalo"] = np.arange(len(meta_ordem))

    colunas = colunas_temporais_evento(X_base)
    if not colunas:
        return X_base

    trabalho = pd.concat([
        meta_ordem[["run_id", "ordem_intervalo"]].reset_index(drop=True),
        X_base[colunas].reset_index(drop=True),
    ], axis=1)
    trabalho["orig_idx"] = np.arange(len(trabalho))
    trabalho = trabalho.sort_values(["run_id", "ordem_intervalo", "orig_idx"])

    novas_partes = [trabalho]
    for lag in lags:
        lagged = trabalho.groupby("run_id")[colunas].shift(lag)
        lagged.columns = [f"{col}_lag{lag}" for col in colunas]
        novas_partes.append(lagged)

    contexto = pd.concat(novas_partes, axis=1)
    for col in colunas:
        lag3 = f"{col}_lag3"
        if lag3 in contexto.columns:
            contexto[f"{col}_trend3"] = contexto[col] - contexto[lag3]

    contexto = contexto.sort_values("orig_idx")
    contexto = contexto.drop(columns=["run_id", "ordem_intervalo", "orig_idx"], errors="ignore")
    contexto = contexto.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([X_base, contexto.drop(columns=colunas, errors="ignore")], axis=1)


def preparar_dataset_evento_proximidade(X_base, y_base, meta_base, target=EVENT_TARGET_DELTA, threshold=EVENT_THRESHOLD_PP):
    if target not in y_base.columns:
        raise ValueError(f"Alvo {target} nao encontrado em y_base.")

    X_evento = adicionar_contexto_temporal_evento(X_base, meta_base)
    y_delta = y_base[target].reset_index(drop=True).astype(float)
    y_evento = (np.abs(y_delta) >= threshold).astype(int)
    meta_evento = meta_base.reset_index(drop=True).copy()
    meta_evento["delta_target"] = y_delta
    meta_evento["evento_proximidade"] = y_evento
    return X_evento, y_delta, y_evento, meta_evento


def avaliar_classificacao_evento(y_true, y_pred, y_score, modelo, split):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if len(np.unique(y_true)) > 1 and np.std(y_score) > 1e-9:
        avg_precision = float(average_precision_score(y_true, y_score))
    else:
        avg_precision = np.nan

    return {
        "modelo": modelo,
        "split": split,
        "event_rate_real": float(np.mean(y_true)),
        "event_rate_pred": float(np.mean(y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_evento": float(precision),
        "recall_evento": float(recall),
        "f1_evento": float(f1),
        "avg_precision": avg_precision,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def avaliar_delta_unico(y_true, y_pred, modelo, split, alvo=EVENT_TARGET_DELTA):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 and np.std(y_true) > 1e-9 and np.std(y_pred) > 1e-9 else np.nan
    return {
        "modelo": modelo,
        "split": split,
        "alvo_delta": alvo,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "corr": corr,
    }



In [ ]:
if "X_mlp" not in globals() or "y_mlp" not in globals() or "meta_mlp" not in globals():
    raiz_mlp = localizar_raiz_projeto_memmap()
    X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)

if len(X_mlp) < 12 or EVENT_TARGET_DELTA not in y_mlp.columns:
    display(Markdown("Dataset insuficiente para o modelo em duas etapas de proximidade."))
else:
    X_evento, y_delta_evento, y_evento, meta_evento = preparar_dataset_evento_proximidade(
        X_mlp, y_mlp, meta_mlp
    )
    split_evento = separar_intervalos(meta_evento)
    if len(split_evento["test"]) == 0:
        split_evento["test"] = split_evento["val"]
    if len(split_evento["val"]) == 0:
        split_evento["val"] = split_evento["test"]

    display(Markdown(
        f"**Dataset evento proximidade:** {len(X_evento)} intervalos, {X_evento.shape[1]} features "
        f"com contexto temporal, alvo `{EVENT_TARGET_DELTA}`, limiar={EVENT_THRESHOLD_PP:.2f} p.p. "
        f"Taxa de evento={float(y_evento.mean()):.1%}. Separacao: {split_evento['modo']}. "
        f"Treino={len(split_evento['train'])}, validacao={len(split_evento['val'])}, teste={len(split_evento['test'])}."
    ))

    resumo_runs_evento = meta_evento.groupby("run_id").agg(
        intervalos=("evento_proximidade", "size"),
        eventos=("evento_proximidade", "sum"),
        taxa_evento=("evento_proximidade", "mean"),
        delta_abs_p90=("delta_target", lambda s: float(np.percentile(np.abs(s), 90))),
    ).reset_index()
    display(resumo_runs_evento)

    Xv_evento = X_evento.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    y_delta_v = y_delta_evento.to_numpy(float)
    y_evento_v = y_evento.to_numpy(int)

    train_idx = split_evento["train"]
    train_event_idx = train_idx[y_evento_v[train_idx] == 1]

    if len(np.unique(y_evento_v[train_idx])) < 2 or len(train_event_idx) < 5:
        display(Markdown(
            "A separacao atual nao tem exemplos suficientes de evento no treino para classificador + regressor. "
            "Diminua `EVENT_THRESHOLD_PP` ou colete mais runs com aproximacao real de obstaculos."
        ))
    else:
        clf_evento = Pipeline([
            ("x_scaler", StandardScaler()),
            ("mlp_evento", MLPClassifier(
                hidden_layer_sizes=(32, 16),
                activation="relu",
                solver="lbfgs",
                alpha=0.05,
                max_iter=2000,
                random_state=42,
            )),
        ])
        clf_dummy = DummyClassifier(strategy="most_frequent")

        reg_evento = Pipeline([
            ("x_scaler", StandardScaler()),
            ("mlp_delta_evento", MLPRegressor(
                hidden_layer_sizes=(64, 16),
                activation="relu",
                solver="lbfgs",
                alpha=0.05,
                max_iter=2000,
                random_state=42,
            )),
        ])
        reg_dummy_evento = DummyRegressor(strategy="median")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_evento.fit(Xv_evento[train_idx], y_evento_v[train_idx])
            reg_evento.fit(Xv_evento[train_event_idx], signed_log1p_array(y_delta_v[train_event_idx]))

        clf_dummy.fit(Xv_evento[train_idx], y_evento_v[train_idx])
        reg_dummy_evento.fit(Xv_evento[train_event_idx], y_delta_v[train_event_idx])

        linhas_classificacao = []
        linhas_regressao = []
        predicoes_evento = {}

        for nome_split, idx in [("validacao", split_evento["val"]), ("teste", split_evento["test"] )]:
            if len(idx) == 0:
                continue

            proba_evento = clf_evento.predict_proba(Xv_evento[idx])[:, 1]
            pred_evento = (proba_evento >= EVENT_PROBA_THRESHOLD).astype(int)
            pred_evento_dummy = clf_dummy.predict(Xv_evento[idx]).astype(int)
            score_dummy = np.full(len(idx), float(np.mean(y_evento_v[train_idx])))

            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento, proba_evento, "MLP evento", nome_split
            ))
            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento_dummy, score_dummy, "Classe majoritaria", nome_split
            ))

            pred_delta_evento = signed_expm1_array(reg_evento.predict(Xv_evento[idx]))
            pred_delta_gate = np.where(pred_evento == 1, pred_delta_evento, 0.0)
            pred_delta_oracle_gate = np.where(y_evento_v[idx] == 1, pred_delta_evento, 0.0)
            pred_delta_dummy_gate = np.where(pred_evento == 1, reg_dummy_evento.predict(Xv_evento[idx]), 0.0)
            pred_delta_zero = np.zeros(len(idx), dtype=float)

            linhas_regressao.extend([
                avaliar_delta_unico(y_delta_v[idx], pred_delta_gate, "MLP evento + MLP delta", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_oracle_gate, "Oracle evento + MLP delta", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_dummy_gate, "MLP evento + mediana evento", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_zero, "Zero delta", nome_split),
            ])

            predicoes_evento[nome_split] = {
                "idx": idx,
                "proba_evento": proba_evento,
                "evento_real": y_evento_v[idx],
                "evento_pred": pred_evento,
                "delta_real": y_delta_v[idx],
                "delta_pred_gate": pred_delta_gate,
                "delta_pred_oracle_gate": pred_delta_oracle_gate,
            }

        resultados_classificacao_evento = pd.DataFrame(linhas_classificacao)
        resultados_regressao_evento = pd.DataFrame(linhas_regressao)

        display(Markdown("**Classificacao do evento de proximidade:**"))
        display(resultados_classificacao_evento)
        px.bar(
            resultados_classificacao_evento,
            x="split", y="recall_evento", color="modelo", barmode="group",
            title="Recall do evento de proximidade por split",
            template="plotly_white",
        ).show()

        display(Markdown("**Regressao do delta apos gate de evento:**"))
        display(resultados_regressao_evento.sort_values(["split", "MAE"]).reset_index(drop=True))
        px.bar(
            resultados_regressao_evento,
            x="split", y="MAE", color="modelo", barmode="group",
            title=f"MAE em {EVENT_TARGET_DELTA}: modelo em duas etapas x baselines",
            template="plotly_white",
        ).show()

        if "teste" in predicoes_evento:
            pred_teste_evento = predicoes_evento["teste"]
            meta_teste_evento = meta_evento.iloc[pred_teste_evento["idx"]].reset_index(drop=True)
            inspecao_evento = pd.DataFrame({
                "run_id": meta_teste_evento["run_id"],
                "sample_id": meta_teste_evento["sample_id"],
                "delta_real": pred_teste_evento["delta_real"],
                "evento_real": pred_teste_evento["evento_real"],
                "proba_evento": pred_teste_evento["proba_evento"],
                "evento_pred": pred_teste_evento["evento_pred"],
                "delta_pred_gate": pred_teste_evento["delta_pred_gate"],
                "delta_pred_oracle_gate": pred_teste_evento["delta_pred_oracle_gate"],
            })
            display(Markdown("**Intervalos de teste mais extremos para inspecao:**"))
            display(inspecao_evento.assign(abs_delta=lambda df: df["delta_real"].abs()).sort_values("abs_delta", ascending=False).drop(columns="abs_delta").head(20))



### Validacao da sincronizacao entre flow, IMU e depth

A pergunta principal é se os intervalos salvos em memmap estao coerentes: `dt_s`, `depth_age_s`, `depth_dt_s`, vetores de optical flow e deltas de depth devem descrever a mesma janela temporal da logica de proximidade visual.

In [ ]:
COLUNAS_SYNC_INTERVALOS = ["dt_s", "depth_age_s", "depth_dt_s", "flow_valid_points", "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad", "pan_comp_delta_rad"]


def montar_dataframe_intervalos_memmap(base_dir):
    runs = carregar_todas_runs_depth_memmap(base_dir)
    if not runs:
        return pd.DataFrame()
    partes = []
    for run in runs:
        df = run["intervalos"].copy()
        df["run_id"] = run["run_dir"].name
        partes.append(df)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def resumir_sincronizacao_intervalos(df):
    if df.empty:
        return pd.DataFrame()
    cols = [c for c in COLUNAS_SYNC_INTERVALOS if c in df.columns]
    return df[cols].describe().T


raiz_flow = localizar_raiz_projeto_memmap()
intervalos_flow_df = montar_dataframe_intervalos_memmap(raiz_flow)

if intervalos_flow_df.empty:
    display(Markdown("Nenhum intervalo memmap encontrado para validar sincronizacao flow/depth."))
else:
    print(f"Intervalos avaliados: {len(intervalos_flow_df)}")
    print(intervalos_flow_df.groupby("run_id").size().rename("intervalos"))
    display(resumir_sincronizacao_intervalos(intervalos_flow_df))
    px.box(intervalos_flow_df, x="run_id", y="dt_s", title="Distribuicao do intervalo temporal entre atualizacoes visuais", labels={"dt_s": "Delta de tempo do intervalo visual (s)", "run_id": "Run"}, template="plotly_white").show()
    px.scatter(intervalos_flow_df, x="radial_flow_p90_px", y="delta_depth_close_5m_pp", color="run_id", size="flow_valid_points", hover_data=["sample_id", "dt_s", "depth_age_s", "pan_comp_delta_rad"], title="Flow radial do intervalo x variacao de proximidade no depth", labels={"radial_flow_p90_px": "P90 do flow radial (px)", "delta_depth_close_5m_pp": "Delta pixels < 5 m (p.p.)"}, template="plotly_white").show()
    serie = intervalos_flow_df.reset_index(drop=True)
    fig_tempo = go.Figure()
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["flow_mag_p90_px"], mode="lines", name="P90 flow"))
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["delta_depth_close_5m_pp"], mode="lines", name="Delta pixels < 5m", yaxis="y2"))
    fig_tempo.update_layout(title="Sequencia dos intervalos: flow visual e delta de proximidade", xaxis_title="Intervalos concatenados em ordem de leitura", yaxis=dict(title="P90 flow (px)"), yaxis2=dict(title="Delta pixels < 5m (p.p.)", overlaying="y", side="right"), template="plotly_white", hovermode="x unified")
    fig_tempo.show()